# Лаб. 3, функція Біла (Beale function)

$$f(x, y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2, \qquad -4.5 \le x, y \le 4.5$$

Глобальний мінімум: $f(3,\ 0.5) = 0$

Генетичний алгоритм (дійсне кодування, особина $= (x, y)$):

1. Початкова популяція: $N$ випадкових точок, рівномірно розподілених в області.
2. Пристосованість: чим менше $f$, тим краща особина.
3. Відбір: турнір. Беремо $k$ випадкових особин і залишаємо найкращу.
4. Схрещування (арифметичне), $\lambda \sim U(0, 1)$:
$$c_1 = \lambda p_1 + (1 - \lambda) p_2, \qquad c_2 = (1 - \lambda) p_1 + \lambda p_2$$
5. Мутація: з імовірністю $p_m$ додаємо до гена шум $\mathcal{N}(0, \sigma^2)$, а $\sigma$ зменшується з кожним поколінням.
6. Елітизм: найкращі $e$ особин переходять у нове покоління без змін.
7. Повторюємо кроки 3–6 протягом $G$ поколінь.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def beale(x, y):
    return (1.5 - x + x * y)**2 + (2.25 - x + x * y**2)**2 + (2.625 - x + x * y**3)**2


lo, hi = -4.5, 4.5


def genetic(N=100, G=100, k=3, pc=0.9, pm=0.2, sigma0=1.0, elite=2, seed=1):
    rng = np.random.default_rng(seed)
    pop = rng.uniform(lo, hi, (N, 2))
    history, best = [], []
    for g in range(G):
        fit = beale(pop[:, 0], pop[:, 1])
        order = np.argsort(fit)
        history.append(pop.copy())
        best.append((pop[order[0]].copy(), fit[order[0]]))

        new = [pop[i] for i in order[:elite]]
        sigma = sigma0 * (1 - g / G)
        while len(new) < N:
            p1, p2 = (pop[min(rng.integers(0, N, k), key=lambda i: fit[i])] for _ in range(2))
            if rng.random() < pc:
                lam = rng.random()
                c1, c2 = lam * p1 + (1 - lam) * p2, (1 - lam) * p1 + lam * p2
            else:
                c1, c2 = p1.copy(), p2.copy()
            for c in (c1, c2):
                mask = rng.random(2) < pm
                c[mask] += rng.normal(0, sigma, mask.sum())
                new.append(np.clip(c, lo, hi))
        pop = np.array(new[:N])
    return history, best

In [2]:
history, best = genetic()
(bx, by), bf = best[-1]
print(f"x = {bx:.6f}, y = {by:.6f}, f = {bf:.3e}")

X, Y = np.meshgrid(np.linspace(lo, hi, 400), np.linspace(lo, hi, 400))
Z = beale(X, Y)
levels = np.logspace(-3, 5.5, 30)

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(X, Y, Z, levels=levels, norm="log", cmap="viridis", extend="both")
ax.plot(3, 0.5, "w*", ms=12, mec="k")
pts = ax.scatter([], [], c="red", s=10, zorder=3)
title = ax.set_title("")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)


def update(g):
    pts.set_offsets(history[g])
    title.set_text(f"покоління {g}, f_min = {best[g][1]:.2e}")
    return pts, title


anim = FuncAnimation(fig, update, frames=len(history), interval=150)
plt.close(fig)
HTML(anim.to_jshtml())

x = 2.996926, y = 0.498891, f = 4.255e-06
